# 04 — Train 3D CNN

**Input dataset:** `lemon-dataset` (attach at `/kaggle/input/lemon-dataset/`)

Trains a shallow 3D CNN with two heads:
- **`wellbeing_output`** — trait anxiety, chronic stress, neuroticism (z-scored regression)
- **`network_output`** — DMN, Salience, ECN activation 0–1 (sigmoid)

Uses 5-fold cross-validation. Best model saved as `best_model.keras`.

**After running:** Save Version → `lemon-model` dataset.
You'll need `best_model.keras` + `wb_stats.json` to serve via FastAPI.

In [ ]:
!pip install -q tensorflow scikit-learn matplotlib

In [ ]:
import tensorflow as tf
print('TF:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

In [ ]:
import numpy as np, json, os, shutil

DATA_DIR = '/kaggle/input/lemon-dataset'

X     = np.load(f'{DATA_DIR}/X.npy')            # (N, 2, 48, 48, 32)
y_reg = np.load(f'{DATA_DIR}/y_regression.npy') # (N, 3)
y_net = np.load(f'{DATA_DIR}/y_network.npy')    # (N, 3)

# TF expects channels-last: (N, 48, 48, 32, 2)
X_tf = np.transpose(X, (0, 2, 3, 4, 1)).astype(np.float32)

print(f'X_tf:  {X_tf.shape}')
print(f'y_reg: {y_reg.shape}')
print(f'y_net: {y_net.shape}')

# Copy wb_stats.json to output so we can save it with the model
shutil.copy(f'{DATA_DIR}/wb_stats.json', '/kaggle/working/wb_stats.json')
shutil.copy(f'{DATA_DIR}/net_scaler.pkl', '/kaggle/working/net_scaler.pkl')
print('Stats files copied to /kaggle/working ✓')

In [ ]:
def augment_batch(X_batch):
    """
    X_batch: (N, 48, 48, 32, 2) channels-last
    Apply random flips and mild noise.
    """
    out = X_batch.copy()
    for i in range(len(out)):
        if np.random.random() < 0.5: out[i] = out[i, ::-1]       # L/R flip
        if np.random.random() < 0.5: out[i] = out[i, :, ::-1]    # A/P flip
        out[i] += np.random.normal(0, 0.02, out[i].shape).astype(np.float32)
    return out

print('Augmentation ready ✓')

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers, regularizers

def build_model():
    reg = regularizers.l2(1e-4)
    inp = keras.Input(shape=(48, 48, 32, 2), name='bold_input')

    x = layers.Conv3D(32, 3, padding='same', activation='relu', kernel_regularizer=reg)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling3D(2)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv3D(64, 3, padding='same', activation='relu', kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling3D(2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv3D(128, 3, padding='same', activation='relu', kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling3D()(x)
    x = layers.Dropout(0.4)(x)

    shared = layers.Dense(64, activation='relu', kernel_regularizer=reg)(x)
    shared = layers.Dropout(0.3)(shared)

    reg_out = layers.Dense(32, activation='relu')(shared)
    reg_out = layers.Dense(3, name='wellbeing_output')(reg_out)

    net_out = layers.Dense(32, activation='relu')(shared)
    net_out = layers.Dense(3, activation='sigmoid', name='network_output')(net_out)

    model = keras.Model(inputs=inp, outputs=[reg_out, net_out])
    return model

build_model().summary()

In [ ]:
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt

BATCH_SIZE = 8
EPOCHS     = 60
N_FOLDS    = 5

kf              = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
best_val_loss   = np.inf
fold_val_losses = []
fold_histories  = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_tf)):
    print(f'\n{"="*50}  FOLD {fold+1}/{N_FOLDS}  {"="*50}')

    X_tr,  X_val  = X_tf[tr_idx],   X_tf[val_idx]
    yr_tr, yr_val = y_reg[tr_idx],  y_reg[val_idx]
    yn_tr, yn_val = y_net[tr_idx],  y_net[val_idx]

    # Augment training set
    X_aug = augment_batch(X_tr)
    X_tr_all  = np.concatenate([X_tr,  X_aug],  axis=0)
    yr_tr_all = np.concatenate([yr_tr, yr_tr],  axis=0)
    yn_tr_all = np.concatenate([yn_tr, yn_tr],  axis=0)

    m = build_model()
    m.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss={'wellbeing_output': 'mse', 'network_output': 'mse'},
        loss_weights={'wellbeing_output': 1.0, 'network_output': 0.5},
        metrics={'wellbeing_output': 'mae', 'network_output': 'mae'},
    )

    history = m.fit(
        X_tr_all,
        {'wellbeing_output': yr_tr_all, 'network_output': yn_tr_all},
        validation_data=(X_val, {'wellbeing_output': yr_val, 'network_output': yn_val}),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=[
            keras.callbacks.EarlyStopping(patience=12, restore_best_weights=True, monitor='val_loss'),
            keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=6, min_lr=1e-5),
        ],
        verbose=1,
    )

    val_loss = min(history.history['val_loss'])
    fold_val_losses.append(val_loss)
    fold_histories.append(history.history)
    print(f'Fold {fold+1} best val_loss: {val_loss:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        m.save('/kaggle/working/best_model.keras')
        print(f'  → New best saved ✓')

print(f'\nFold val losses: {[round(v,4) for v in fold_val_losses]}')
print(f'Mean: {np.mean(fold_val_losses):.4f} ± {np.std(fold_val_losses):.4f}')

In [ ]:
best_fold = int(np.argmin(fold_val_losses))
h = fold_histories[best_fold]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(h['loss'], label='train'); axes[0].plot(h['val_loss'], label='val')
axes[0].set_title(f'Total loss — fold {best_fold+1}'); axes[0].legend()
axes[1].plot(h['wellbeing_output_mae'], label='train')
axes[1].plot(h['val_wellbeing_output_mae'], label='val')
axes[1].set_title('Wellbeing MAE (z-scored)'); axes[1].legend()
plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=100)
plt.show()

print()
print('OUTPUT FILES:')
for fname in os.listdir('/kaggle/working'):
    path = f'/kaggle/working/{fname}'
    print(f'  {fname}  ({os.path.getsize(path)/1e6:.1f} MB)')
print()
print('NEXT STEPS:')
print('  Save Version → "Save as Dataset" → name it "lemon-model"')
print('  Then open notebook 05 to validate with Grad-CAM')